# Baseline Pricing Strategies

Evaluate three simple baselines for predicting a trade's price within the day:

- **first**: use the first trade price of the day as the predictor for all subsequent trades
- **best**: use the daily mean (an oracle / upper bound)
- **last**: use the price of the most recent prior day's last trade

MAE (mean absolute error) is reported for both `DOLLAR_PRICE` and `YIELD`. Any model has to beat these baselines to be useful.

The original implementation looped per (CUSIP, date) in Python with one query each. This notebook computes everything with two BigQuery window-function queries.

In [ ]:
from google.cloud import bigquery
import pandas as pd

client = bigquery.Client(project='nyu-datasets')
TABLE = 'nyu-datasets.munibonds.trades_typed'

## Pick the active universe

Same liquidity criteria as `02_descriptive_analysis.ipynb`: many dealer trades, many trade days, frequent enough to support time-series. We pick the top-N most active for benchmarking.

In [ ]:
TOP_N = 500
MIN_DAILY_TRADES = 20  # only score days with at least this many dealer trades

UNIVERSE_QUERY = f"""
SELECT CUSIP
FROM `{TABLE}`
WHERE TRADE_TYPE_INDICATOR = 'D'
  AND DOLLAR_PRICE IS NOT NULL
  AND TRADE_DATE IS NOT NULL
GROUP BY CUSIP
HAVING COUNT(*) > 1000 AND COUNT(DISTINCT TRADE_DATE) > 100
ORDER BY COUNT(*) DESC
LIMIT {TOP_N}
"""

universe_cusips = client.query(UNIVERSE_QUERY).to_dataframe()['CUSIP'].tolist()
print(f'Top {len(universe_cusips):,} most-active dealer-traded CUSIPs')

## Per-trade MAE for first/best/last predictors

For each dealer trade in our universe, attach the three predictors using window functions:
- **first** = first price of the day (`FIRST_VALUE` ordered by `TIME_OF_TRADE`)
- **best** = mean price of the day
- **last** = price of last trade on the most recent prior day (`LAG` over CUSIP, ordered by trade time across days)

Compute absolute error of each predictor against the actual trade price (and yield).

In [ ]:
PER_TRADE_QUERY = f"""
WITH dealer AS (
  SELECT CUSIP, TRADE_DATE, TIME_OF_TRADE,
         DOLLAR_PRICE, YIELD,
         CONCAT(CAST(TRADE_DATE AS STRING), ' ', CAST(TIME_OF_TRADE AS STRING)) AS dt_key
  FROM `{TABLE}`
  WHERE TRADE_TYPE_INDICATOR = 'D'
    AND DOLLAR_PRICE IS NOT NULL
    AND CUSIP IN UNNEST(@cusips)
),
with_predictors AS (
  SELECT *,
    -- first trade of the day
    FIRST_VALUE(DOLLAR_PRICE) OVER (
      PARTITION BY CUSIP, TRADE_DATE ORDER BY TIME_OF_TRADE, dt_key
    ) AS price_first,
    FIRST_VALUE(YIELD) OVER (
      PARTITION BY CUSIP, TRADE_DATE ORDER BY TIME_OF_TRADE, dt_key
    ) AS yield_first,
    -- best (oracle): same-day mean
    AVG(DOLLAR_PRICE) OVER (PARTITION BY CUSIP, TRADE_DATE) AS price_best,
    AVG(YIELD)        OVER (PARTITION BY CUSIP, TRADE_DATE) AS yield_best,
    -- last (most recent prior dealer trade)
    LAST_VALUE(IF(TRADE_DATE < TRADE_DATE, NULL, NULL)) OVER () AS _ignored,
    LAG(DOLLAR_PRICE) OVER (PARTITION BY CUSIP ORDER BY dt_key) AS lag_price,
    LAG(YIELD)        OVER (PARTITION BY CUSIP ORDER BY dt_key) AS lag_yield,
    LAG(TRADE_DATE)   OVER (PARTITION BY CUSIP ORDER BY dt_key) AS lag_date
  FROM dealer
),
scored AS (
  SELECT CUSIP, TRADE_DATE, DOLLAR_PRICE, YIELD,
         price_first, price_best,
         IF(lag_date < TRADE_DATE, lag_price, NULL) AS price_last,
         yield_first, yield_best,
         IF(lag_date < TRADE_DATE, lag_yield, NULL) AS yield_last
  FROM with_predictors
)
SELECT CUSIP, TRADE_DATE,
       COUNT(*) AS num_trades,
       AVG(ABS(DOLLAR_PRICE - price_first)) AS price_mae_first,
       AVG(ABS(DOLLAR_PRICE - price_best))  AS price_mae_best,
       AVG(ABS(DOLLAR_PRICE - price_last))  AS price_mae_last,
       AVG(ABS(YIELD - yield_first)) AS yield_mae_first,
       AVG(ABS(YIELD - yield_best))  AS yield_mae_best,
       AVG(ABS(YIELD - yield_last))  AS yield_mae_last
FROM scored
GROUP BY CUSIP, TRADE_DATE
HAVING num_trades >= @min_daily_trades
"""

job_config = bigquery.QueryJobConfig(
    query_parameters=[
        bigquery.ArrayQueryParameter('cusips', 'STRING', universe_cusips),
        bigquery.ScalarQueryParameter('min_daily_trades', 'INT64', MIN_DAILY_TRADES),
    ]
)
per_day = client.query(PER_TRADE_QUERY, job_config=job_config).to_dataframe()
print(f'{len(per_day):,} (CUSIP, day) cells with at least {MIN_DAILY_TRADES} dealer trades')
per_day.head()

## Average MAE across the active universe

In [ ]:
# Per-CUSIP averages
by_cusip = (
    per_day.groupby('CUSIP')
    [['price_mae_first', 'price_mae_best', 'price_mae_last',
      'yield_mae_first', 'yield_mae_best', 'yield_mae_last',
      'num_trades']]
    .mean()
    .round(3)
)
by_cusip.head(10)

In [ ]:
print('Universe-wide average MAE (in same units as price/yield):')
summary = per_day[
    ['price_mae_first', 'price_mae_best', 'price_mae_last',
     'yield_mae_first', 'yield_mae_best', 'yield_mae_last']
].mean().round(4)
summary.to_frame('MAE')

## Interpretation

- `*_best` is an oracle (uses same-day mean), so it's the lower bound an in-day model could approach.
- `*_first` is the realistic naive baseline a real-time predictor must beat.
- `*_last` is the inter-day baseline: yesterday's last price.

A useful pricing model needs to outperform `*_first` for in-day prediction or `*_last` for end-of-day-to-next-day prediction.